<a href="https://www.kaggle.com/code/mrafraim/dl-day-46-saving-loding-models?scriptVersionId=309136397" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Day 46: Saving & Loading Models

Welcome to Day 46!

Today you'll learn:
- Save trained CNN weights correctly
- Load them in a separate environment/script
- Verify correctness (not assume)

If you found this notebook helpful, your **<b style="color:skyblue;">UPVOTE</b>** would be greatly appreciated! It helps others discover the work and supports continuous improvement.

---

# The Hidden Problem

Most people:

✔ Save model  
✔ Load model  
✔ Assume it works  

But never verify.

### Real Risk

- Wrong architecture → silent mismatch
- Wrong preprocessing → wrong predictions
- Device mismatch → runtime errors

### Principle

Saving/loading is NOT trivial.

It is:

👉 Reconstructing the exact learned function

# What Exactly Are We Saving?

Two Ways to Save

**1. Save `state_dict` (RECOMMENDED)**

- Saves only weights
- Flexible
- Production standard

**2. Save entire model (NOT recommended)**

- Saves architecture + weights
- Breaks if code changes

```python
# Correct approach
torch.save(model.state_dict(), "cnn_weights.pth")
```

# Saving Best Model (From Training Loop)

Final epoch ≠ best performance

We save:

👉 best validation model

**Pattern**

if val improves → save  
else → ignore

```python
if val_loss < best_val_loss:
    best_val_loss = val_loss
    torch.save(model.state_dict(), "best_model.pth")
```

# Separate Script Mindset

Loading should happen in a DIFFERENT script.

Why?

→ Simulates real deployment


### Requirement

You MUST recreate:
- Model architecture
- Same class definition

Loading a model is NOT:

> “open file and use model”

It is:

Rebuild structure
        +
Inject learned parameters
        =
Working model

# Rebuild Model Architecture

Architecture must match EXACTLY.

If not:

❌ shape mismatch<br>
❌ incorrect behavior

```python
class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*7*7, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

```

# Load Weights Properly

Correct Loading Steps

1. Initialize model
2. Load weights
3. Move to device
4. Set eval mode

```python
device = "cuda" if torch.cuda.is_available() else "cpu"

model = CNN()
model.load_state_dict(torch.load("best_model.pth"))
model = model.to(device)
model.eval()
```

# Device Compatibility (COMMON BUG)

### Problem

Model saved on GPU, loaded on CPU → error

### Solution

Use map_location

```python
model.load_state_dict(torch.load("best_model.pth", map_location=device))
```

# Inference Function (Reusable)

**Purpose**

Encapsulate prediction logic.

Reusable in:
- API
- scripts
- batch inference

```python
def predict(image, model):
    model.eval()

    with torch.no_grad():
        output = model(image)
        pred = output.argmax(dim=1)

    return pred
```

# Verification
You must confirm:

Loaded model = original model

**Strategy**

Compare predictions:
- Before saving
- After loading

```python
# BEFORE saving (store output)
model.eval()
with torch.no_grad():
    original_output = model(image)

# AFTER loading
model_loaded = CNN()
model_loaded.load_state_dict(torch.load("best_model.pth"))
model_loaded.eval()

with torch.no_grad():
    loaded_output = model_loaded(image)

# Compare
print(torch.allclose(original_output, loaded_output, atol=1e-6))
```

# Interpretation of Verification

### If TRUE

✔ Model loaded correctly  
✔ Pipeline is consistent  

### If FALSE

Something is wrong:

- Architecture mismatch
- Preprocessing mismatch
- Device inconsistency

# Common Mistakes
❌ Saving entire model object  
❌ Changing architecture after saving  
❌ Forgetting model.eval()  
❌ Ignoring preprocessing consistency  
❌ Not verifying outputs  

> Most deployment bugs come from THIS stage.

# Practical Workflow
1. Train model  
2. Save best weights  
3. Create new script  
4. Rebuild architecture  
5. Load weights  
6. Run inference  
7. Verify correctness  

This is EXACTLY how production pipelines work.

# Key Takeaways from Day 46
- Always save state_dict (not full model)
- Architecture must match exactly
- Use `map_location` for portability
- Always call `model.eval()` for inference
- Verification is mandatory, not optional
- This step bridges training → deployment

---

<p style="text-align:center; color:skyblue; font-size:18px;">
© 2026 Mostafizur Rahman
</p>
